In [2]:

import pandas as pd
import datetime as dt
import numpy as np
import re

# Load the dataset
df = pd.read_csv('../data/online_retail.csv')

# Explore the data
df.info()
df.describe()
df.shape

# Clean the data
df.columns = [re.sub(r'([a-z])([A-Z])', r'\1_\2', col).lower() for col in df.columns]  # standardize column names
df.drop_duplicates(inplace=True)
df.dropna(subset=['customer_id','invoice_no'], inplace=True)
df['description'] = df['description'].str.title()  # capitalize product names
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['customer_id'] = df['customer_id'].astype(int)
df['month'] = df['invoice_date'].dt.month

df['sales'] = df['quantity'] * df['unit_price']
df = df[(df.select_dtypes(include='number') >= 0).all(axis=1)]  # remove negative values

# Calculate snapshot date for recency
snapshot_date = df['invoice_date'].max() + dt.timedelta(days=1)

# Calculate RFM values
rfm = df.groupby('customer_id').agg({
    'invoice_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'invoice_no': 'count',                                      # Frequency
    'sales': 'sum'                                              # Monetary
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']
print('MIN:{} MAX:{}'.format(df['invoice_date'].min(), df['invoice_date'].max()))

# Assign quartiles for R, F, M
r_quartiles = pd.qcut(rfm['Recency'], 4, labels=range(4,0,-1), duplicates='drop')
f_quartiles = pd.qcut(rfm['Frequency'], 4, labels=range(1,5), duplicates='drop')
m_quartiles = pd.qcut(rfm['Monetary'], 4, labels=range(1,5), duplicates='drop')

rfm = rfm.assign(
    R=r_quartiles.values.astype(int),
    F=f_quartiles.values.astype(int),
    M=m_quartiles.values.astype(int)
)

# Create RFM Segment and Score
rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Score'] = rfm[['R','F','M']].sum(axis=1)

# General Segment based on RFM_Score
conditions = [
    rfm['RFM_Score'] >= 9,
    (rfm['RFM_Score'] >= 7) & (rfm['RFM_Score'] < 9),
    (rfm['RFM_Score'] >= 5) & (rfm['RFM_Score'] < 7),
    rfm['RFM_Score'] < 5
]
choices = ['Champions', 'Loyal Customers', 'At Risk', 'Lost']
rfm['General_Segment'] = np.select(conditions, choices, default='Lost')

# Reorder columns
rfm = rfm[['Recency','Frequency','Monetary','R','F','M','RFM_Segment','RFM_Score','General_Segment']]

# Export datasets for Power BI
rfm.reset_index().to_csv('rfm_segments.csv', index=False)  # Customer-level data
df.to_csv('cleaned_online_retail_data.csv', index=False)  # Transaction-level data

print("✅ Exported 'rfm_segments.csv' and 'cleaned_online_retail_data.csv' ready for Power BI")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
MIN:2010-12-01 08:26:00 MAX:2011-12-09 12:50:00
✅ Exported 'rfm_segments.csv' and 'cleaned_online_retail_data.csv' ready for Power BI


MIN:2010-12-01 08:26:00 MAX;2011-12-09 12:50:00 


,Recency,Frequency,Monetary,R,F,M,RFM_Segment,RFM_Score,General_Segment
customer_id,,,,,,,,,
12346,326,1,77183.60,1,1,4,114,6,At Risk
12347,2,182,4310.00,4,4,4,444,12,Champions
12348,75,31,1797.24,2,2,4,224,8,Loyal Customers
12349,19,73,1757.55,3,3,4,334,10,Champions
12350,310,17,334.40,1,1,2,112,4,Lost
...,...,...,...,...,...,...,...,...,...
18280,278,10,180.60,1,1,1,111,3,Lost
18281,181,7,80.82,1,1,1,111,3,Lost
18282,8,12,178.05,4,1,1,411,6,At Risk


np.int64(0)